In [12]:
##### Area of Applicability (AOA) analysis
# calculates the dissimilarity index (DI) and AOA flag for every raster cell, following Meyer & Pebesma (2021)

from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rio
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [13]:
##### SET-UP

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from helpers.step_thin_sample import ThinConfig, prepare_thinned_dataset

DATA_DIR = ROOT / "Data"
AOA_DIR = ROOT / "Results" / "AOA"
AOA_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
##### LOAD DATA

raster_model = pd.read_parquet(DATA_DIR / "raster_predictor_matrix_final.parquet")
folds_df = pd.read_csv(DATA_DIR / "spatial_CV_folds.csv")

imp_cap_df = pd.read_csv(ROOT / "Results/RF_models/capital_rf_spatial_CV/feature_importance.csv")
imp_lab_df = pd.read_csv(ROOT / "Results/RF_models/labor_rf_spatial_CV/feature_importance.csv")

ref_path = DATA_DIR / "total_production_tonnes_2020.tif"   # reference grid for rasterizing

In [15]:
##### THIN TRAINING DATA (same method as RF training, so AOA reflects only data used in the models)

THIN_CONFIGS = [
    ThinConfig(name="capital", raw_dataset="training_dataset_capital_RF.csv"),
    ThinConfig(name="labor",   raw_dataset="training_dataset_labor_RF.csv"),
]

thinned_datasets = {tc.name: prepare_thinned_dataset(tc, DATA_DIR) for tc in THIN_CONFIGS}
capital_model = thinned_datasets["capital"]
labor_model = thinned_datasets["labor"]


── capital ──────────────────────────────
country_ID  n_before  n_after  n_dropped  thinned
       BRA      3684      852       2832     True
       USA      2630     1053       1577     True
       CAN      1380      300       1080     True
       ARG       396      259        137     True
       TUR        80       77          3     True
       CHN        31       31          0    False
       DEU        13       13          0    False
       EGY        26       26          0    False
       ESP        17       17          0    False
       FRA        22       22          0    False
       BEL         2        2          0    False
       GRC         4        4          0    False
       HRV         2        2          0    False
       ITA        20       20          0    False
       MEX        24       24          0    False
       PRT         3        3          0    False
       ROU         8        8          0    False
       TZA        24       24          0    False
       

In [16]:
##### ADD CV FOLD ASSIGNMENTS

# converts wide fold columns (1 = test in that fold) into a single fold_id per country
def wide_folds_to_fold_id(fold_df, prefix, id_col="country_ID"):
    fold_cols = [c for c in fold_df.columns if c.startswith(f"{prefix}_fold_")]
    if not fold_cols:
        raise ValueError(f"No fold columns found for prefix '{prefix}'")

    vals = fold_df[fold_cols].fillna(0).values
    fold_id = vals.argmax(axis=1) + 1
    fold_id = np.where(vals.sum(axis=1) == 0, np.nan, fold_id)   # countries not assigned to any fold

    out = fold_df[[id_col]].copy()
    out["fold_id"] = fold_id
    return out

fold_id_cap = wide_folds_to_fold_id(folds_df, "capital")
fold_id_lab = wide_folds_to_fold_id(folds_df, "labor")

capital_model["country_ID"] = capital_model["PROJ_ID"].astype(str).str[:3]
labor_model["country_ID"] = labor_model["PROJ_ID"].astype(str).str[:3]

capital_model = capital_model.merge(fold_id_cap, on="country_ID", how="left")
labor_model = labor_model.merge(fold_id_lab, on="country_ID", how="left")

In [17]:
##### PREDICTOR COLUMNS

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_cereals_share_base_100_production_tonnes',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_vegetables_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_share_base_100_large_field',
       'rtv_log_share_base_100_with_nightlights']

labor_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_rest_of_crops_share_base_100_production_tonnes',
       'rtv_log_sugar_crops_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_pct_base_100_GDP_ag', 'rtv_log_share_base_100_large_field',
       'rtv_log_pct_base_100_cropland_irrigated']

In [18]:
##### PREP PREDICTOR MATRICES

X_train_lab_full = labor_model[labor_cols + ["fold_id"]].dropna()
X_train_lab = X_train_lab_full[labor_cols]
fold_ids_lab = X_train_lab_full["fold_id"].values

X_train_cap_full = capital_model[capital_cols + ["fold_id"]].dropna()
X_train_cap = X_train_cap_full[capital_cols]
fold_ids_cap = X_train_cap_full["fold_id"].values

X_pixel_raw_lab = raster_model[labor_cols]
X_pixel_raw_cap = raster_model[capital_cols]

valid_mask_lab = X_pixel_raw_lab.notna().all(axis=1)
valid_mask_cap = X_pixel_raw_cap.notna().all(axis=1)

scaler_lab = StandardScaler()
X_train_scaled_lab = scaler_lab.fit_transform(X_train_lab)
X_pixel_scaled_lab = scaler_lab.transform(X_pixel_raw_lab[valid_mask_lab])

scaler_cap = StandardScaler()
X_train_scaled_cap = scaler_cap.fit_transform(X_train_cap)
X_pixel_scaled_cap = scaler_cap.transform(X_pixel_raw_cap[valid_mask_cap])

In [19]:
##### VARIABLE IMPORTANCE WEIGHTING (Meyer & Pebesma, 2021, Section 2.2)

imp_lab_map = imp_lab_df.set_index("feature")["permutation_mean"].to_dict()
imp_cap_map = imp_cap_df.set_index("feature")["permutation_mean"].to_dict()

importance_lab = np.array([imp_lab_map[c] for c in labor_cols])
importance_cap = np.array([imp_cap_map[c] for c in capital_cols])

X_train_scaled_lab = X_train_scaled_lab * importance_lab
X_pixel_scaled_lab = X_pixel_scaled_lab * importance_lab

X_train_scaled_cap = X_train_scaled_cap * importance_cap
X_pixel_scaled_cap = X_pixel_scaled_cap * importance_cap

In [20]:
##### DISSIMILARITY INDEX + AOA THRESHOLD

def cross_fold_nn_distance(X_scaled, fold_ids):
    """Distance from each training point to its nearest neighbor among
    training points NOT in the same CV fold (needed for the AOA threshold)."""
    dists = np.full(len(X_scaled), np.nan)
    fold_ids = np.asarray(fold_ids)
    for fold in np.unique(fold_ids):
        held_out_mask = fold_ids == fold
        pool_mask = ~held_out_mask
        nn = NearestNeighbors(n_neighbors=1, n_jobs=-1)
        nn.fit(X_scaled[pool_mask])
        d, _ = nn.kneighbors(X_scaled[held_out_mask])
        dists[held_out_mask] = d[:, 0]
    return dists

def compute_di_and_threshold(X_train_scaled, X_pixel_scaled, fold_ids):
    nn_train = NearestNeighbors(n_neighbors=2, n_jobs=-1)
    nn_train.fit(X_train_scaled)
    train_nn_dists, _ = nn_train.kneighbors(X_train_scaled)
    avg_train_nn_dist = train_nn_dists[:, 1].mean()

    nn_pixel = NearestNeighbors(n_neighbors=1, n_jobs=-1)
    nn_pixel.fit(X_train_scaled)
    pixel_nn_dists, _ = nn_pixel.kneighbors(X_pixel_scaled)
    DI = pixel_nn_dists[:, 0] / avg_train_nn_dist

    train_cross_fold_dists = cross_fold_nn_distance(X_train_scaled, fold_ids)
    train_di = train_cross_fold_dists / avg_train_nn_dist
    q25, q75 = np.percentile(train_di, [25, 75])
    threshold = q75 + 1.5 * (q75 - q25)

    return DI, threshold

DI_lab, aoa_threshold_lab = compute_di_and_threshold(X_train_scaled_lab, X_pixel_scaled_lab, fold_ids_lab)
DI_cap, aoa_threshold_cap = compute_di_and_threshold(X_train_scaled_cap, X_pixel_scaled_cap, fold_ids_cap)

print(f"Capital AOA threshold: {aoa_threshold_cap:.3f} | {(DI_cap <= aoa_threshold_cap).mean():.1%} of valid cells inside AOA")
print(f"Labor AOA threshold:   {aoa_threshold_lab:.3f} | {(DI_lab <= aoa_threshold_lab).mean():.1%} of valid cells inside AOA")

Capital AOA threshold: 2.447 | 88.0% of valid cells inside AOA
Labor AOA threshold:   2.157 | 63.6% of valid cells inside AOA


In [21]:
##### RASTERIZE DI / AOA

ref_da = rio.open_rasterio(ref_path).squeeze()
height, width = ref_da.shape
transform = ref_da.rio.transform()

def values_to_dataarray(values, rows, cols, template, height, width):
    grid = np.full((height, width), np.nan, dtype=np.float32)
    grid[rows, cols] = values
    return template.copy(data=grid)

def get_rowcol(valid_mask):
    rows, cols = rasterio.transform.rowcol(
        transform,
        raster_model.loc[valid_mask, "x"].values,
        raster_model.loc[valid_mask, "y"].values,
    )
    return np.array(rows), np.array(cols)

rows_lab, cols_lab = get_rowcol(valid_mask_lab)
rows_cap, cols_cap = get_rowcol(valid_mask_cap)

di_grid_labor = values_to_dataarray(DI_lab, rows_lab, cols_lab, ref_da, height, width)
aoa_grid_labor = values_to_dataarray(
    (DI_lab <= aoa_threshold_lab).astype(np.float32), rows_lab, cols_lab, ref_da, height, width
)

di_grid_capital = values_to_dataarray(DI_cap, rows_cap, cols_cap, ref_da, height, width)
aoa_grid_capital = values_to_dataarray(
    (DI_cap <= aoa_threshold_cap).astype(np.float32), rows_cap, cols_cap, ref_da, height, width
)

In [22]:
##### SAVE OUTPUTS

outputs = {
    "DI_capital.tif":  di_grid_capital,
    "AOA_capital.tif": aoa_grid_capital,
    "DI_labor.tif":    di_grid_labor,
    "AOA_labor.tif":   aoa_grid_labor,
}

for fname, da in outputs.items():
    da = da.astype("float32").rio.write_nodata(np.nan)
    da.rio.to_raster(AOA_DIR / fname, compress="deflate")
    print(f"Saved {fname}")

# AOA thresholds
thresholds = {
    "capital": float(aoa_threshold_cap),
    "labor":   float(aoa_threshold_lab),
}
(AOA_DIR / "aoa_thresholds.json").write_text(json.dumps(thresholds, indent=2))

Saved DI_capital.tif
Saved AOA_capital.tif
Saved DI_labor.tif
Saved AOA_labor.tif


66